# Part 3: NLP Pipeline & Sequence Modeling

## Task 1: Dataset Understanding

In [1]:
import pandas as pd

df = pd.read_csv('Data_Sets/part_3_nlp_sequence_modeling/customer_support_text_classification.csv')
print(f"Dataset shape: {df.shape}")
print(f"Number of records: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Dataset shape: (1500, 6)
Number of records: 1500

Columns: ['ticket_id', 'channel', 'customer_message', 'sentiment_label', 'word_count', 'urgent_flag']

Data types:
ticket_id             str
channel               str
customer_message      str
sentiment_label       str
word_count          int64
urgent_flag         int64
dtype: object

Missing values:
ticket_id           0
channel             0
customer_message    0
sentiment_label     0
word_count          0
urgent_flag         0
dtype: int64


In [2]:
# Sample text records

print("Sample text records:")
for label in df['sentiment_label'].unique():
    sample = df[df['sentiment_label'] == label]['customer_message'].iloc[0]
    print(f"\n[{label.upper()}]: {sample}")

Sample text records:

[NEUTRAL]: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.

[POSITIVE]: The refund process was fast and convenient. I appreciate the quick response.

[NEGATIVE]: My refund is still pending and this experience is frustrating. My ticket number is 33927.


In [3]:
# Target labels/classes and Class distribution

print("Target labels/classes:", df['sentiment_label'].unique())
print(f"Number of classes: {df['sentiment_label'].nunique()}")
print(f"\nClass distribution:")
print(df['sentiment_label'].value_counts())
print(f"\nClass distribution (%):")
print(df['sentiment_label'].value_counts(normalize=True).round(3) * 100)

Target labels/classes: <StringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str
Number of classes: 3

Class distribution:
sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64

Class distribution (%):
sentiment_label
neutral     34.9
negative    33.1
positive    31.9
Name: proportion, dtype: float64


**Observations:**
1. The dataset has 1500 records with 3 sentiment classes.
2. Classes are fairly balanced (neutral: ~35%, negative: ~33%, positive: ~32%).
3. There are no missing values in the dataset
4. Messages are relatively short customer support texts.

## Task 2: Text Preprocessing

In [4]:
import numpy as np
import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

df['clean_text'] = df['customer_message'].apply(preprocess_text)

print("Before vs After preprocessing:")
for i in range(3):
    print(f"\nOriginal : {df['customer_message'].iloc[i]}")
    print(f"Cleaned  : {df['clean_text'].iloc[i]}")

Before vs After preprocessing:

Original : I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Cleaned  : need information payment process ticket number please respond soon possible

Original : I need information about the payment process.
Cleaned  : need information payment process

Original : The refund process was fast and convenient. I appreciate the quick response.
Cleaned  : refund process fast convenient appreciate quick response


**Preprocessing steps applied:**
1. **Lowercasing** — normalizes all text to lowercase
2. **Removing special characters** — keeps only alphabetic characters and spaces (removes ticket numbers, punctuation, etc.)
3. **Tokenization** — splits text into individual words using NLTK's word_tokenize
4. **Stopword removal** — removes common English words that don't carry much meaning (the, is, at, etc.)
5. **Short token removal** — removes single-character tokens

## Task 3: Text Vectorization
**Converting text into numerical format** using:
1. Bag of Words
2. TF-IDF

In [5]:
# Splitting the data
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 1200, Test size: 300


In [11]:
# Method 1: Bag of Words (BoW)
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print(f"Method 1: Bag of Words (BoW)")
print("-----------------------------")
print(f"BoW - Train shape: {X_train_bow.shape}, Test shape: {X_test_bow.shape}")
print(f"BoW - Vocabulary size: {len(bow_vectorizer.vocabulary_)}")
print(f"BoW - Sample feature names: {list(bow_vectorizer.vocabulary_.keys())[:10]}")

# Method 2: Term Frequency-Inverse Document Frequency (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer 

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\nMethod 2: Term Frequency-Inverse Document Frequency (TF-IDF)")
print("-------------------------------------------------------------")
print(f"TF-IDF - Train shape: {X_train_tfidf.shape}, Test shape: {X_test_tfidf.shape}")
print(f"TF-IDF - Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"\nSample TF-IDF values for first document (non-zero entries):")
first_doc = X_train_tfidf[0].toarray().flatten()
non_zero_idx = first_doc.nonzero()[0]
feature_names = tfidf_vectorizer.get_feature_names_out()
for idx in non_zero_idx[:5]:
    print(f"  '{feature_names[idx]}': {first_doc[idx]:.4f}")


Method 1: Bag of Words (BoW)
-----------------------------
BoW - Train shape: (1200, 146), Test shape: (300, 146)
BoW - Vocabulary size: 146
BoW - Sample feature names: ['confirm', 'whether', 'ticket', 'assigned', 'please', 'respond', 'soon', 'possible', 'unhappy', 'account']

Method 2: Term Frequency-Inverse Document Frequency (TF-IDF)
-------------------------------------------------------------
TF-IDF - Train shape: (1200, 146), Test shape: (300, 146)
TF-IDF - Vocabulary size: 146

Sample TF-IDF values for first document (non-zero entries):
  'assigned': 0.4525
  'confirm': 0.4525
  'please': 0.2660
  'possible': 0.2982
  'respond': 0.2982


## Task 4: Baseline Model
**Building a simple baseline model** using:
1. Naive Bayes with Bag of Words
2. Logistic Regression with TF-IDF

Evaluating the models

In [12]:
# Model 1: Naive Bayes Classifier with BoW
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
nb_preds = nb_bow.predict(X_test_bow)

print(f"\nModel 1: Naive Bayes Classifier with BoW")
print("-----------------------------------------")

print(f"Accuracy: {accuracy_score(y_test, nb_preds):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, nb_preds))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, nb_preds))


Model 1: Naive Bayes Classifier with BoW
-----------------------------------------
Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        99
     neutral       1.00      1.00      1.00       105
    positive       1.00      1.00      1.00        96

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Confusion Matrix:
[[ 99   0   0]
 [  0 105   0]
 [  0   0  96]]


In [13]:
# Model 2: Logistic Regression with TF-IDF
from sklearn.linear_model import LogisticRegression

lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
lr_preds = lr_tfidf.predict(X_test_tfidf)

print(f"\nModel 2: Logistic Regression with TF-IDF")
print("-----------------------------------------")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_preds))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_preds))


Model 2: Logistic Regression with TF-IDF
-----------------------------------------
Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        99
     neutral       1.00      1.00      1.00       105
    positive       1.00      1.00      1.00        96

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Confusion Matrix:
[[ 99   0   0]
 [  0 105   0]
 [  0   0  96]]


**Inference:**
1. Both "Naive Bayes with Bag of Words" AND "Logistic Regression with TF-IDF" are outputing predictions with 100% accuracy.